# Capital Asset Pricing Model (CAPM)

## 0. Preliminaries
### 0.1 Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cvxpy as cp

from pathlib import Path

### 0.2 Plot Settings

In [ ]:
%matplotlib ipympl

## 0.3 Helper Function

In [ ]:
def mean_std(df):
    """Mean and standard deviation of daily returns."""
    returns = df.pct_change().dropna()
    n_days = len(returns)
    return returns.mean() * n_days, returns.std() * np.sqrt(n_days)

## 1. Data
### 1.1 Load
Run `reinvest extract transform load` to actually get the data first! Then point to the `workdir`you chose.

In [ ]:
workdir = '..'  # Modify as needed!

timeseries = pd.read_parquet(Path(workdir) / 'data/proc/Timeseries.parquet')
overview = pd.read_parquet(Path(workdir) / 'data/proc/Overview.parquet').set_index('isin')

### 1.2 Define Portfolio Constituents
CAPM does not tell you what to invest in.

In [ ]:
overview

In [ ]:
isins = [
    'GB00B15KY989',  # Broad Commodities
    'DE000A0S9GB0',  # Gold
    'IE00B5BMR087',  # S&P 500
    'LU0328475792',  # STOXX Europe 600
    'LU0274209740',  # Japan
    'IE00B4L5YC18',  # Emerging Markets
    'LU1287023185',  # Euro Govt. bonds 7-10 yrs.
    'IE00B3VWN518'   # USD Govt. bonds 7-10 yrs.
]
index = overview.loc[isins]['asset'] + ' ' + overview.loc[isins]['region']
n_assets = len(isins)
risk_free = timeseries['LU0290358497']  # Money Market (Overnight Swap)
world = timeseries['IE00B4L5Y983']      # MSCI World for comparison
selection = timeseries[isins].dropna()

### 1.3 Inspect Portfolio
Pick one!

In [ ]:
year = 2025

#### 1.3.1 Assets

In [ ]:
normalized = 100 * selection.loc[f'{year}'] / selection.loc[f'{year}'].iloc[0]

fig, ax = plt.subplots(figsize=(10, 6))
normalized.plot(ax=ax)
ax.set_title('Contituent Overview')
ax.set_ylabel(f'Normalized Asset Value (100 @ {year}-01-01)')
ax.axhline(100, c='k', lw=0.5, ls='--')
plt.pause(0.05)
fig.tight_layout()

#### 1.3.2 Returns
If $p_t$ is the asset price on day $t$, then simple (daily) returns $r_t$ are defined as:

$$
r_t = \frac{p_t - p_{t-1}}{p_{t-1}}
$$

In [ ]:
returns = selection.loc[f'{year}'].pct_change().dropna()
rf_mean, rf_std = mean_std(risk_free.loc[f'{year}'])
world_mean, world_std = mean_std(world.loc[f'{year}'])
i = 0

In [ ]:
fig, ax = plt.subplots()
returns[isins[i]].plot(ax=ax, label=index.iloc[i])
ax.axhline(c='k', lw=0.5, ls='--')
ax.set_ylabel('Simple Returns')
ax.set_title('Returns approx. a Random Walk')
ax.legend()
plt.pause(0.05)
fig.tight_layout()

i += 1

## 2. Exploit Anti-Correlation
The variance of a linear combination of random variables can be smaller than the variance of any one random variable ([Wiki](https://en.wikipedia.org/wiki/Variance#Linear_combinations)), _but only if some elements of the covariance are negative_!

In [ ]:
returns.corr().style.background_gradient(cmap='RdYlGn', vmin=-1, vmax=1).format('{:.2f}').map(
    lambda val: 'background-color: white' if val >= 0.99 else ''
)

### 2.1 Constrained Optimization
Maximimize return by varying portfolio composition!

In [ ]:
means = returns.mean().values * len(returns)
sigma = returns.cov().values * len(returns)
max_std = np.sqrt(np.diag(sigma)[means.argmax()])

weights_ = cp.Variable(n_assets)
risk_ = cp.quad_form(weights_, sigma)

objective_ = cp.Minimize(risk_)
contraints_ = [
    cp.sum(weights_) == 1.0,
    weights_ >= 0.0
]
problem_ = cp.Problem(objective_, contraints_)
min_stddev = np.sqrt(problem_.solve())

stddevs = np.linspace(min_stddev, max_std, 101)

rewards, risks, sharpes, allocations = [], [], [], []

for stddev in stddevs:
    weights = cp.Variable(n_assets)
    risk = cp.quad_form(weights, sigma)
    reward = means @ weights
    
    objective = cp.Maximize(reward)
    contraints = [
        risk <= stddev**2,
        cp.sum(weights) == 1.0,
        weights >= 0.0
    ]
    
    problem = cp.Problem(objective, contraints)
    problem.solve()

    if weights.value is not None:
        allocation = weights.value
        portfolio_reward = means @ allocation
        portfolio_risk = np.sqrt(allocation @ sigma @ allocation)
        sharpe = (portfolio_reward - rf_mean) / (portfolio_risk - rf_std)

        allocations.append(allocation)
        rewards.append(portfolio_reward)
        risks.append(portfolio_risk)
        sharpes.append(sharpe)
    else:
        print(f'The targeted standard deviation of {stddev} is infeasable!')
        continue

rewards = np.array(rewards)
risks = np.array(risks)
sharpes = np.array(sharpes)
allocations = np.array(allocations)

### 2.2 Efficiency Frontier

In [ ]:
best = sharpes.argmax()

tangent_return = rewards[best]
tangent_risk = risks[best]
max_sharpe = sharpes[best]
max_risk = risks[risks.argmax()]

cml_risks = np.linspace(0, max_risk, 100)
cml_returns = rf_mean + max_sharpe * (cml_risks - rf_std)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(risks, rewards, label='Efficient Frontier')
ax.scatter([tangent_risk], [tangent_return], c='C3', marker='*', s=100, label='Tangency Portfolio', zorder=4)
ax.scatter([world_std], [world_mean], c='C3', marker='X', s=100, label='MSCI World')
ax.scatter([rf_std], [rf_mean], c='C2', marker='x', s=100, label='Risk Free')
sc = ax.scatter(risks, rewards, c=sharpes, cmap='viridis', label='Sharpe Ratio', zorder=1)
ax.plot(cml_risks, cml_returns, c='k', ls='--', lw=1.0, label='Capital Market Line', zorder=3)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('Sharpe Ratio')

ax.set_xlabel('Annualized Risk')
ax.set_ylabel('Annualized Return')
ax.set_title('Efficient Frontier and Capital Market Line')
ax.grid(True, ls='--')
ax.legend()
plt.pause(0.05)
fig.tight_layout()

### 2.3 Strategic Allocations
Optimal asset allocation depends on your risk appetite!

In [ ]:
i = 0

In [ ]:
portfolio = pd.DataFrame(allocations[i*10], index=index, columns=['weight'])
fig, ax = plt.subplots(figsize=(8, 4))
portfolio.plot.barh(y='weight', ax=ax, zorder=3)
ax.grid(True, axis='x')
ax.set_xlabel('Relative Weight')
ax.set_title(f'Strategic allocation for risk level {i*10} %')
plt.pause(0.05)
fig.tight_layout()
i += 1